# Classifying deconvolved calcium with a 3-state state-space model

This notebook mirrors the sorted-spikes classification tutorial, but uses OASIS-deconvolved calcium events at 30 Hz.

The classifier uses three discrete states:

1. **Local**: smooth position evolution (`RandomWalk`)
2. **Stationary**: position stays in one bin (`Identity`)
3. **Jump**: non-local transitions (`Uniform`)

The executable workflow in this notebook stays on the KDE path (`SortedSpikesClassifier` fed with deconvolved spikes). Package-level `CalciumClassifier` support for gamma and ZIG likelihoods is implemented separately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from replay_trajectory_classification import (
    DiagonalDiscrete,
    Environment,
    Identity,
    RandomWalk,
    SortedSpikesClassifier,
    Uniform,
)
from replay_trajectory_classification.calcium_sorted_spikes_decoding import (
    deconvolve_and_binarize,
)
from replay_trajectory_classification.simulate_calcium import (
    make_continuous_replay,
    make_fragmented_replay,
    make_hover_replay,
    make_simulated_run_data,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_context("talk", font_scale=0.8)
np.random.seed(0)

In [ ]:
TRACK_HEIGHT = 120.0
N_NEURONS = 12
PLACE_FIELD_MEANS = np.linspace(0.0, TRACK_HEIGHT, N_NEURONS)
SAMPLING_FREQUENCY = 30
INTERNAL_SAMPLING_FREQUENCY = 1000
RUNNING_SPEED = 10.0
SIGMA = 1.0
N_RUNS = 2
STATE_NAMES = ["Local", "Stationary", "Jump"]

TARGET_STATE_DURATION_S = 0.5
DIAGONAL_VALUE = 1.0 - 1.0 / (TARGET_STATE_DURATION_S * SAMPLING_FREQUENCY)

# A direct speed prior gives ~0.11 cm^2/frame at 10 cm/s and 30 Hz.
# We round up to one-bin variance for a numerically stable local state.
RAW_MOVEMENT_SD = RUNNING_SPEED / SAMPLING_FREQUENCY
RAW_MOVEMENT_VAR = RAW_MOVEMENT_SD**2
MOVEMENT_VAR = max(RAW_MOVEMENT_VAR, 1.0)
PLACE_BIN_SIZE = 1.0

print(f"Diagonal stay probability: {DIAGONAL_VALUE:.3f}")
print(f"Raw movement variance prior: {RAW_MOVEMENT_VAR:.3f} cm^2/frame")
print(f"Notebook movement variance used: {MOVEMENT_VAR:.3f} cm^2/frame")

## Simulate run data and fit the 3-state classifier

The key calcium-timescale adjustments are:

- The **discrete-state persistence** is chosen from a target duration of about 0.5 s, rather than copied from ephys defaults.
- The **local-motion prior** starts from the expected frame-to-frame displacement at 30 Hz, then is rounded up to a one-bin variance so the Local state is distinguishable from the exact-bin Stationary state.

In [ ]:
def make_run_dataset(seed):
    time, position, sampling_frequency, calcium_traces, true_spikes, _ = make_simulated_run_data(
        sampling_frequency=SAMPLING_FREQUENCY,
        internal_sampling_frequency=INTERNAL_SAMPLING_FREQUENCY,
        track_height=TRACK_HEIGHT,
        running_speed=RUNNING_SPEED,
        n_runs=N_RUNS,
        place_field_means=PLACE_FIELD_MEANS,
        sigma=SIGMA,
        rng=np.random.default_rng(seed),
    )
    inferred_spikes = deconvolve_and_binarize(calcium_traces)
    return {
        "time": time,
        "position": position,
        "sampling_frequency": sampling_frequency,
        "calcium_traces": calcium_traces,
        "true_spikes": true_spikes,
        "inferred_spikes": inferred_spikes,
    }


def make_classifier():
    return SortedSpikesClassifier(
        environments=[Environment(environment_name="", place_bin_size=PLACE_BIN_SIZE)],
        continuous_transition_types=[
            [RandomWalk(movement_var=MOVEMENT_VAR), Uniform(), Uniform()],
            [Uniform(), Identity(), Uniform()],
            [Uniform(), Uniform(), Uniform()],
        ],
        discrete_transition_type=DiagonalDiscrete(DIAGONAL_VALUE),
        sorted_spikes_algorithm="spiking_likelihood_kde",
        sorted_spikes_algorithm_params={
            "position_std": 3.0,
            "use_diffusion": False,
            "block_size": None,
        },
    )


def state_probability(results, posterior_name="acausal_posterior"):
    return getattr(results, posterior_name).sum("position")


def plot_state_probability(ax, results, posterior_name, title):
    posterior = state_probability(results, posterior_name)
    for state in posterior.state.values:
        ax.plot(posterior.time.values, posterior.sel(state=state).values, label=str(state), linewidth=2)
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Probability")
    ax.set_title(title)
    ax.legend(loc="upper right", frameon=False)


train = make_run_dataset(0)
test = make_run_dataset(1)

classifier = make_classifier()
classifier.fit(train["position"], train["inferred_spikes"])
run_results = classifier.predict(test["inferred_spikes"], time=test["time"], state_names=STATE_NAMES)

In [ ]:
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(14, 10), constrained_layout=True)

spike_time_ind, neuron_ind = np.nonzero(test["inferred_spikes"])
axes[0].scatter(test["time"][spike_time_ind], neuron_ind + 1, s=5, color="black")
axes[0].set_ylabel("Neuron")
axes[0].set_title("Held-out run: deconvolved spikes")

run_results.causal_posterior.sum("state").plot(
    x="time", y="position", ax=axes[1], cmap="bone_r", add_colorbar=False
)
axes[1].plot(test["time"], test["position"], color="magenta", linestyle="--", linewidth=2)
axes[1].set_title("Held-out run: causal position posterior")
axes[1].set_ylabel("Position")

run_results.acausal_posterior.sum("state").plot(
    x="time", y="position", ax=axes[2], cmap="bone_r", add_colorbar=False
)
axes[2].plot(test["time"], test["position"], color="magenta", linestyle="--", linewidth=2)
axes[2].set_title("Held-out run: acausal position posterior")
axes[2].set_ylabel("Position")

plot_state_probability(axes[3], run_results, "acausal_posterior", "Held-out run: acausal state probabilities")
axes[3].set_xlabel("Time [s]")

sns.despine(offset=5)

## Classify continuous, stationary, and fragmented calcium events

The calcium simulation helpers now use durations that are meaningful at 30 Hz, so the events below span tens of frames rather than collapsing to 1-3 frames.

In [ ]:
def make_event_dataset(name, event_fn, seed):
    time, true_spikes, calcium_traces = event_fn(
        sampling_frequency=SAMPLING_FREQUENCY,
        internal_sampling_frequency=INTERNAL_SAMPLING_FREQUENCY,
        place_field_means=PLACE_FIELD_MEANS,
        sigma=SIGMA,
        rng=np.random.default_rng(seed),
    )
    inferred_spikes = deconvolve_and_binarize(calcium_traces)
    results = classifier.predict(inferred_spikes, time=time, state_names=STATE_NAMES)
    return {
        "name": name,
        "time": time,
        "true_spikes": true_spikes,
        "calcium_traces": calcium_traces,
        "inferred_spikes": inferred_spikes,
        "results": results,
    }


event_datasets = [
    make_event_dataset("Continuous", make_continuous_replay, 2),
    make_event_dataset("Stationary", make_hover_replay, 3),
    make_event_dataset("Fragmented", make_fragmented_replay, 4),
]

In [ ]:
fig, axes = plt.subplots(len(event_datasets), 2, figsize=(14, 10), constrained_layout=True)

for row_ind, dataset in enumerate(event_datasets):
    dataset["results"].acausal_posterior.sum("state").plot(
        x="time", y="position", ax=axes[row_ind, 0], cmap="bone_r", add_colorbar=False
    )
    axes[row_ind, 0].set_title(f"{dataset['name']} event: acausal position posterior")
    axes[row_ind, 0].set_ylabel("Position")
    plot_state_probability(
        axes[row_ind, 1],
        dataset["results"],
        "acausal_posterior",
        f"{dataset['name']} event: acausal state probabilities",
    )

for ax in axes[-1, :]:
    ax.set_xlabel("Time [s]")

sns.despine(offset=5)

## Note on ZIG support

This notebook intentionally keeps the executable path on deconvolved spikes + KDE so the workflow stays close to the sorted-spikes tutorial.

The package now also exposes `CalciumClassifier`, which supports both:

- `calcium_likelihood` (gamma)
- `deconv_calcium_likelihood` (ZIG)

Those likelihood-specific classifier paths are covered in the package code and unit tests rather than executed here.